In [ ]:
"""
SV callset matching with Truvari-like algorithm
modified to work with breakpoints
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import importlib
from pathlib import Path

import pandas as pd

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as utils
import svensemble.chunk_match as chunk_match

importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(utils)
importlib.reload(chunk_match)

pd.set_option('display.max_columns', None)

preprocessed_savedir = Path('saved/preprocessed/')
matched_savedir = Path('saved/matched/')

try:
    os.makedirs(matched_savedir)
    print(f'Created {matched_savedir.absolute()}')
except FileExistsError:
    print(f'{matched_savedir.absolute()} already exists')

insert_seq_source = {
    'gnomad': 'svlen',
    'gridss': 'alt',
    'manta': 'svlen'
}


In [27]:
gridss_bp_df = pd.read_csv(preprocessed_savedir / 'gridss_bps.csv')
gnomad_bp_df = pd.read_csv(preprocessed_savedir / 'gnomad_bps.csv')
manta_diploid_bp_df = pd.read_csv(preprocessed_savedir / 'manta_diploid_bps.csv')
bp_df_dict = {'gridss': gridss_bp_df, 'gnomad': gnomad_bp_df, 'manta': manta_diploid_bp_df}

In [ ]:
matched_bp_df = chunk_match.get_connected_pairwise_matches_df(
    bp_df_dict=bp_df_dict,
    insert_size_threshold=0.7,
    max_dist=10
)

matched_bp_df

In [ ]:
info_event_fields = [f'{caller}_INFO_EVENT' for caller in bp_df_dict.keys()]
(~matched_bp_df[info_event_fields].isna()).value_counts()


In [ ]:
joint_bp_df = matched_bp_df.copy()

ref_callset = 'gnomad'

for caller, df in bp_df_dict.items():
    # Skip reference callset
    # since we do not need the full reference callset for training
    # just the fact that calls match to it
    if caller == ref_callset:
        continue

    # Add suffix to columns
    df_suffixed = df.add_prefix(f'{caller}_')
    # Join with matched_bp_df
    joint_bp_df = joint_bp_df.merge(
        df_suffixed,
        left_on=f'{caller}_INFO_EVENT',
        right_on=f'{caller}_INFO_EVENT',
        how='outer'
    )

joint_bp_df

In [ ]:
(~joint_bp_df[info_event_fields].isna()).value_counts()

In [ ]:
na_masks = {
    callset_name: joint_bp_df[f'{callset_name}_INFO_EVENT'].isna() for callset_name in bp_df_dict.keys()
}

for caller, mask in na_masks.items():
    print(f'{caller} has {mask.sum()} NA values')

# %%

In [ ]:
joint_bp_df.to_csv(matched_savedir / 'joint_bp_df.csv', index=False)